# Darknode AI — 13B foundation trial (QLoRA)

The **fast/cheap trial**: fine-tune the WhiteRabbitNeo-13B security base into Darknode with LoRA, on the always-available CC0 authored + synthetic data — **no large downloads**. This proves the whole pipeline end to end (clean data → QLoRA → adapter → Modelfile) before you spend on the 100B/500B runs.

**Runtime:** pick a GPU runtime (Runtime → Change runtime type → GPU). A **free T4** works with the small overrides below; an **A100** (Colab Pro) runs the full preset. Then **Run All**.

Adapter is saved to Google Drive so it survives a disconnect.

In [ ]:
# 1) GPU check
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU — set Runtime → GPU")

In [ ]:
# 2) Get the repo (private) + install the foundation deps.
# Paste a GitHub PAT with read access when prompted — it is NOT stored.
import getpass, os, subprocess
REPO = "github.com/Darknode-Official/darknode-ai.git"
BRANCH = "feat/foundation-rag-scaling"
if not os.path.isdir("darknode-ai"):
    tok = getpass.getpass("GitHub PAT (read access to darknode-ai): ").strip()
    url = f"https://{tok}@{REPO}" if tok else f"https://{REPO}"
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, "darknode-ai"], check=True)
    del tok, url
%cd darknode-ai
!pip -q install -e . && pip -q install -r requirements-foundation.txt
print("deps installed")

In [ ]:
# 3) Build clean-provenance SFT data (authored + synthetic, both CC0 — no downloads).
#    To add real security datasets, download them to JSONL and append e.g.
#    --dataset whiterabbitneo=wrn.jsonl  (see FOUNDATION.md; provenance guard applies).
!python -m darknode_ai.foundation.dataprep --out data/sft
!echo '--- train/val sizes ---' && wc -l data/sft/train.jsonl data/sft/val.jsonl

In [ ]:
# 4) (optional) Mount Drive so the adapter survives a disconnect.
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/darknode-13b-trial"
print("adapter ->", OUT)

In [ ]:
# 5) Run the 13B QLoRA fine-tune (now ~1254 clean examples after the dataprep fix).
#   T4 (free): ~81s/step; ~120 steps = ~3 epochs = ~2.7h. Checkpoints to Drive
#   every 20 steps and --resume continues from the newest one, so a Colab
#   disconnect is not fatal: just re-run this cell to pick up where it left off.
#   A100 (Pro): drop the overrides to run the full preset much faster.
!python -m darknode_ai.foundation.finetune --preset 13b --i-have-a-gpu \
    --data data/sft --out "$OUT" \
    --batch 1 --grad-accum 4 --seq-len 512 \
    --max-steps 120 --save-steps 20 --resume

In [ ]:
# 6) Package for Ollama: convert the LoRA adapter to a GGUF + emit a Modelfile.
#    Light path (no 26GB merge): FROM the Ollama base + ADAPTER the gguf + persona.
#    Clones llama.cpp on first run. Then, on a box with Ollama and the base pulled:
#      ollama create darknode -f "$OUT/Modelfile"   &&   ollama run darknode
!python -m darknode_ai.foundation.to_gguf \
    --adapter "$OUT" --out "$OUT/darknode-lora.gguf" \
    --llama-cpp /content/llama.cpp \
    --base-model-id WhiteRabbitNeo/WhiteRabbitNeo-13B-v1 \
    --emit-modelfile --ollama-base jimscard/whiterabbit-neo
!echo '--- adapter + gguf files ---' && ls -la "$OUT"